# NCKH 02 — Evaluation Metrics and Horizon Sensitivity

This notebook generates the Section 4.2 figures. Macro-F1 is the
primary multiclass metric; balanced accuracy, per-class recall and
the PR-AUC for the combined `CRITICAL + HIGH_RISK` group are reported
because high-risk misses matter for a decision-support workflow.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

ROOT = Path.cwd()
if not (ROOT / "reports" / "nckh_online_retail_multiclass").exists():
    ROOT = Path("..").resolve()
REPORT = ROOT / "reports" / "nckh_online_retail_multiclass"
SELECTED = REPORT / "selected"
FIG_DIR = REPORT / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["CRITICAL", "HIGH_RISK", "MEDIUM_RISK", "LOW_RISK"]
CLASS_COLORS = {
    "CRITICAL": "#B42318",
    "HIGH_RISK": "#E76F00",
    "MEDIUM_RISK": "#176B87",
    "LOW_RISK": "#2A9D8F",
}
MODEL_LABELS = {
    "recent_zero_rule": "Recent-zero rule",
    "recent_quantile_rule": "Recent-sales quantile rule",
    "logistic_regression": "Logistic regression",
    "random_forest": "Random forest",
    "lightgbm": "LightGBM",
}
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.family": "DejaVu Sans",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
})

with open(SELECTED / "dataset_profile.json", encoding="utf-8") as handle:
    profile = json.load(handle)
with open(REPORT / "selected_config.json", encoding="utf-8") as handle:
    selected_config = json.load(handle)
with open(SELECTED / "metrics.json", encoding="utf-8") as handle:
    metrics = json.load(handle)

In [ ]:
sensitivity = pd.read_csv(REPORT / "horizon_sensitivity.csv")
model_names = list(metrics["models"])
comparison = pd.DataFrame({
    MODEL_LABELS[name]: {
        "Macro-F1": metrics["models"][name]["test"]["macro_f1"],
        "Balanced accuracy": metrics["models"][name]["test"]["balanced_accuracy"],
        "High-risk recall": metrics["models"][name]["test"]["high_risk_group_recall"],
        "High-risk PR-AUC": metrics["models"][name]["test"]["high_risk_pr_auc"],
    } for name in model_names
}).T
comparison

## Figure 5 — Baseline and model comparison

In [ ]:
metric_names = ["Macro-F1", "Balanced accuracy", "High-risk recall", "High-risk PR-AUC"]
plot_df = comparison[metric_names]
fig, ax = plt.subplots(figsize=(10.5, 5.2))
x = np.arange(len(plot_df))
width = 0.19
colors = ["#264653", "#2A9D8F", "#E76F00", "#B42318"]
for index, metric_name in enumerate(metric_names):
    bars = ax.bar(x + (index - 1.5) * width, plot_df[metric_name],
                  width, label=metric_name, color=colors[index])
    ax.bar_label(bars, fmt="%.2f", padding=2, fontsize=7)
ax.set_xticks(x, plot_df.index, rotation=18, ha="right")
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.set_title("Test-period comparison on the selected horizon")
ax.legend(frameon=False, ncol=2, loc="upper left")
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
fig.savefig(FIG_DIR / "05_model_metric_comparison.png", bbox_inches="tight")
plt.show()

## Figure 6 — High-risk precision–recall curves

In [ ]:
fig, ax = plt.subplots(figsize=(7.8, 5.2))
for name in model_names:
    curve = metrics["models"][name]["test"]["high_risk_pr_curve"]
    ax.plot(curve["recall"], curve["precision"],
            linewidth=2, label=MODEL_LABELS[name])
prevalence = metrics["label_prevalence"]["test"]["CRITICAL"] + metrics["label_prevalence"]["test"]["HIGH_RISK"]
ax.axhline(prevalence, color="#6C757D", linestyle="--", linewidth=1,
           label=f"Test high-risk prevalence ({prevalence:.1%})")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel("Recall of CRITICAL + HIGH_RISK")
ax.set_ylabel("Precision of CRITICAL + HIGH_RISK")
ax.set_title("High-risk prioritisation: precision–recall")
ax.legend(frameon=False, fontsize=8)
ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(FIG_DIR / "06_high_risk_precision_recall.png", bbox_inches="tight")
plt.show()

## Figure 7 — Horizon sensitivity

In [ ]:
fig, ax = plt.subplots(figsize=(8.2, 4.8))
ax.plot(sensitivity["horizon_days"], sensitivity["validation_lightgbm_macro_f1"],
        marker="o", linewidth=2.5, color="#176B87", label="Validation Macro-F1")
ax.plot(sensitivity["horizon_days"], sensitivity["test_lightgbm_macro_f1"],
        marker="s", linewidth=2, color="#E76F00", label="Test Macro-F1")
selected_h = selected_config["selected_horizon_days"]
ax.axvline(selected_h, color="#B42318", linestyle="--",
           label=f"Selected horizon = {selected_h} days")
ax.set_xlabel("Future-sales horizon (days)")
ax.set_ylabel("LightGBM Macro-F1")
ax.set_title("Horizon sensitivity and validation-based selection")
ax.set_xticks(sensitivity["horizon_days"])
ax.set_ylim(0, 1)
ax.legend(frameon=False)
ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(FIG_DIR / "07_horizon_sensitivity.png", bbox_inches="tight")
plt.show()

**Interpretation boundary.** The selected horizon is chosen using
validation Macro-F1. Test results are reported only after that
selection. A higher score means better agreement with the defined
future-sales proxy, not proof of lower inventory or improved business
outcomes.